<a href="https://colab.research.google.com/github/Pranayshukla0610/Data-Science-Projects/blob/main/Story_Generator_using_NLP%2C_Transformers%2C_and_Generative_AI_%7C_End_to_End_NLP_Pipeline_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install datasets
!pip install nltk
!pip install spacy


In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import re
import nltk
import spacy

from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec
from wordcloud import WordCloud

from transformers import pipeline
from transformers import AutoTokenizer

from datasets import load_dataset

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

In [ ]:
data = load_dataset('roneneldan/TinyStories')

In [ ]:
train_df = pd.DataFrame(data['train'])

train_df = train_df.sample(3000, random_state=42)

train_df = train_df[['text']]

train_df.head()

In [ ]:
train_df.shape

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.info()

In [ ]:
train_df['story_length'] = train_df['text'].apply(len)

train_df['story_length'].describe()

In [ ]:
plt.figure(figsize=(16,10))
plt.hist(train_df['story_length'], bins=50)
plt.xlabel('story length')
plt.ylabel('frequency')
plt.title('story length distribution')
plt.show()

In [ ]:
train_df['clean_text'] = train_df['text'].str.lower()

In [ ]:
def clean_text(text):
  text = re.sub(r'[a-zA-Z\\s]','',text)
  return text

train_df['clean_text'] = train_df['clean_text'].apply(clean_text)

In [ ]:
train_df['clean_text'] = train_df['clean_text'].apply(lambda x:re.sub(r'\\s+','',x))

In [ ]:
train_df['tokens'] = train_df['clean_text'].apply(lambda x:x.split())

In [ ]:
train_df['tokens'].head()

In [ ]:
stopwords = set(stopwords.words('english'))

def remove_stopwords(tokens):
  return[word for word in tokens if word not in stopwords]

train_df['filtered_tokens'] = train_df['tokens'].apply(remove_stopwords)

In [ ]:
stemmer = PorterStemmer()

def stem_words(tokens):

    return [stemmer.stem(word) for word in tokens]

train_df['stemmed_tokens'] = train_df['filtered_tokens'].apply(stem_words)

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):

    return [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

train_df['lemmatized_tokens'] = train_df['filtered_tokens'].apply(
    lemmatize_words
)

In [ ]:
import nltk

nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
sample_tokens = train_df['tokens'].iloc[0]

nltk.pos_tag(sample_tokens[:20])

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
nlp = spacy.load("en_core_web_sm")

sample_story = train_df['text'].iloc[0]

doc = nlp(sample_story)

for ent in doc.ents:
  print(ent.text, ent.label_)

In [ ]:
all_words = []

for tokens in train_df['filtered_tokens']:
  all_words.extend(tokens)

word_freq = Counter(all_words)

word_freq.most_common(20)

In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400
).generate(' '.join(all_words))

plt.figure(figsize=(16,10))
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
corpus = train_df['clean_text'].head(1000)

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(corpus)
print(X_bow.shape)

In [ ]:
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(corpus)

print(X_tfidf.shape)

In [ ]:
sentences = train_df['lemmatized_tokens'].tolist()


sentences = sentences[:2000]

word2vec_model = Word2Vec(

    sentences,

    vector_size=50,

    window=5,

    min_count=2,

    workers=2
)

In [ ]:
train_df['clean_text'].head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

sample = 'Once Upon a Time'

tokens = tokenizer.encode(sample)

print(tokens)

print(tokenizer.decode(tokens))

In [ ]:
story_generator = pipeline(
    'text-generation',
    model='gpt2'
)

In [ ]:
story = story_generator(

    "Once upon a time in a magical kingdom",

    max_length=100,

    num_return_sequences=1
)

print(story[0]['generated_text'])

In [ ]:
genre = "Fantasy"

prompt = f"""

Write a {genre} story
about a dragon and hidden treasure."""

story = story_generator(
    prompt,
    max_length=100,
    temperature=0.8
)

print(story[0]['generated_text'])

In [ ]:
prompt = """

Create a fantasy character.

Name:
Personality:
Abilities:
Backstory:

"""

character = story_generator(

    prompt,

    max_length=150
)

print(character[0]['generated_text'])

In [ ]:
prompt = """

The old castle door slowly opened,
and suddenly a strange whisper came from inside.

"""

continuation = story_generator(

    prompt,

    max_length=150,

    temperature=0.9
)

print(continuation[0]['generated_text'])

In [ ]:
story = story_generator(

    prompt,

    max_length=150,

    num_beams=5,

    early_stopping=True
)

print(story[0]['generated_text'])

In [ ]:
stories = story_generator(

    prompt,

    max_length=120,

    num_return_sequences=3
)

for i, story in enumerate(stories):

    print(f"Story {i+1}")

    print(story['generated_text'])

    print("-"*50)

In [ ]:
stories = story_generator(

    prompt,

    max_length=120,

    num_return_sequences=3
)

for i, story in enumerate(stories):

    print(f"Story {i+1}")

    print(story['generated_text'])

    print("-"*50)

In [ ]:
story_generator.model.save_pretrained("story_model")

tokenizer.save_pretrained("story_model")

In [ ]:
story_generator.model.save_pretrained("story_model")

tokenizer.save_pretrained("story_model")

In [ ]:
!pip install streamlit pyngrok

In [ ]:
%%writefile app.py

import streamlit as st

from transformers import pipeline

story_generator = pipeline(
    "text-generation",
    model="gpt2"
)

st.title("AI Story Generator")

genre = st.selectbox(
    "Select Genre",
    ["Fantasy", "Horror", "Sci-Fi"]
)

prompt = st.text_area("Enter Prompt")

if st.button("Generate"):

    final_prompt = f"{genre} story: {prompt}"

    story = story_generator(
        final_prompt,
        max_length=150
    )

    st.write(story[0]['generated_text'])

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501